# First bike-rental prediction

We want to predict how many bikes were rented in one hour.

Here is one real row from `data/bike_rentals_mini.csv`:
- hour: 16 means 4 PM
- hour_scaled: 0.696, which is 16 / 23 rounded for display
- temperature: 0.30
- humidity: 0.49
- wind speed: 0.2537
- working_day: 1 means it was a working day
- rental_count: 83 bikes
- target_scaled: 0.083, which means 83 / 1000

## Scaling: keep the real meaning

The dataset keeps the original `rental_count` because people understand bike counts.

The notebook also uses `target_scaled`, which is the same count divided by `1000`. A model could calculate with `83`, but `0.083` keeps the first loss and nudge examples small enough to read.

Whenever we want the human meaning back, we multiply by `1000` again.

In [1]:
rental_count = 83
target_scaled = rental_count / 1000

print(f"target_scaled: {target_scaled:.3f}")
print(f"back to rentals: {target_scaled * 1000:.1f}")

target_scaled: 0.083
back to rentals: 83.0


## Make a deliberately simple first prediction

Our first prediction will not use the weather columns yet. We will guess `0.100`, which means about `100` rentals.

The real target is `0.083`, which means `83` rentals. Now we can measure how wrong the guess was before we try to improve it.

In [2]:
target_scaled = 0.083
prediction_scaled = 0.100

error = prediction_scaled - target_scaled
loss = error * error

print(f"prediction: {prediction_scaled:.3f}")
print(f"target: {target_scaled:.3f}")
print(f"error: {error:.3f}")
print(f"squared error loss: {loss:.6f}")

prediction: 0.100
target: 0.083
error: 0.017
squared error loss: 0.000289


## Read the first loss

The `error` is `prediction - target`.

- Positive error means the prediction was too high.
- Negative error means the prediction was too low.
- Squared-error loss multiplies the error by itself, so the loss is always positive.

Here, an error of `0.017` means the guess is about `17` rentals too high. The goal is not to make the first guess perfect. The goal is to get one clear number that tells us whether a later change helped.

## Introduce one adjustable number

Now we stop guessing a fixed prediction directly and use a tiny model:

```text
prediction = hour_scaled * weight
```

`hour_scaled` comes from the real row. `weight` is adjustable, which means we are allowed to change it and see whether the loss gets smaller.

This is not a good bike-rental model yet. It is a tiny example with one moving part, so the effect of each nudge is easy to see.

In [3]:
hour_scaled = 0.6956521739130435
target_scaled = 0.083

weight = 0.10

prediction = hour_scaled * weight
error = prediction - target_scaled
loss = error * error

print(f"prediction scaled: {prediction:.6f}")
print(f"prediction rentals: {prediction * 1000:.1f}")
print(f"target rentals: {target_scaled * 1000:.1f}")
print(f"loss: {loss:.8f}")

prediction scaled: 0.069565
prediction rentals: 69.6
target rentals: 83.0
loss: 0.00018049


## Nudge the weight and compare

A **nudge** is a small change to the adjustable number.

The next cell tries nearby weights: `0.09`, `0.10`, `0.11`, `0.12`, and `0.13`. Compare their losses and look for two things:

1. which weight has the smallest loss;
2. what happens when the weight goes past that best nearby value.

Be careful with decimal places: `0.012` is not a small step away from `0.12`; it is ten times smaller. A tiny-looking decimal change can make a much smaller prediction and a much larger loss.

In [9]:
hour_scaled = 0.6956521739130435
target_scaled = 0.083


for weight in [0.09, 0.10, 0.11, 0.12, 0.13]:
    prediction = hour_scaled * weight
    error = prediction - target_scaled
    loss = error * error

    print(f"weight: {weight:.2f}")
    print(f"prediction rentals: {prediction * 1000:.1f}")
    print(f"error: {error:.6f}")
    print(f"loss: {loss:.8f}")
    print()

weight: 0.09
prediction rentals: 62.6
error: -0.020391
loss: 0.00041581

weight: 0.10
prediction rentals: 69.6
error: -0.013435
loss: 0.00018049

weight: 0.11
prediction rentals: 76.5
error: -0.006478
loss: 0.00004197

weight: 0.12
prediction rentals: 83.5
error: 0.000478
loss: 0.00000023

weight: 0.13
prediction rentals: 90.4
error: 0.007435
loss: 0.00005528



## Name the idea: gradient

We tried small changes to `weight` and watched the loss.

When increasing `weight` made the loss smaller, that gave us a direction signal: move the weight up from where we started. When the loss rose again, it showed that moving up forever is not the goal.

A **gradient** is a more systematic way to get a local direction signal without trying many guesses by hand.

## Course boundary

For the core path of this course, we use standard-library Python for the neural-network code.

That means no NumPy, pandas, PyTorch, JAX, or autograd in the main implementation. This boundary keeps the learning visible: we will write the prediction, loss, and update steps ourselves.

Later, PyTorch can appear only as an optional comparison after the from-scratch idea is clear.

## Running the project with uv

From the repository root:

```bash
uv sync
uv run pytest
uv run jupyter lab
```

`uv sync` installs the project tools. `uv run pytest` runs the automated checks. `uv run jupyter lab` opens the notebooks.

## Tiny course map

This first notebook starts with one real prediction, one loss, and one adjustable number.

Later notebooks will build from the same pattern:

1. understand slopes from tiny nudges;
2. send learning signals backward through small expressions;
3. build a tiny `.backward()` engine;
4. train a one-hidden-layer MLP on the bike-rental data;
5. prove the gradients and predictions with checks.

The examples will get larger, but the loop stays familiar: predict, measure wrongness, use a direction signal, and update.

## HITL checkpoint

Before moving on, explain these words using the bike-rental row from this notebook:

- **prediction:** what did the model guess?
- **loss:** how did the notebook measure wrongness?
- **nudge:** which weight changes did we try?
- **gradient:** what direction signal did the nudge experiment reveal?

Use the real numbers from the notebook, not only definitions.